In [5]:
import functions
import importlib
importlib.reload(functions)

from functions import *

#### One satellite

In [ ]:
##
##to real
##
def sample_chip_sequence_to_if_file(
    chips_01,
    out_file = "gps_signal_if_float32.dat",
    chip_rate = 1.023e6,
    fs = 4.0e6,
    f_if = 1.25e6,                                           # Defined to this frequency in GNSS SDR
    chunk_size = 10_000_000,
):
    chips_01 = np.asarray(chips_01, dtype=np.uint8)
    chips = np.where(chips_01 == 0, -1.0, 1.0).astype(np.float32)

    duration = len(chips) / chip_rate
    n_samples = int(np.floor(duration * fs))

    with open(out_file, "wb") as f:
        for start in range(0, n_samples, chunk_size):
            stop = min(start + chunk_size, n_samples)

            sample_idx = np.arange(start, stop, dtype=np.int64)
            chip_index = (sample_idx * chip_rate / fs).astype(np.int64)
            chip_index = np.clip(chip_index, 0, len(chips) - 1)

            baseband = chips[chip_index]  # sampled chip waveform, float32
            t = sample_idx / fs
            carrier = np.cos(2 * np.pi *f_if* t).astype(np.float32)

            signal = baseband * carrier
            signal.astype(np.float32).tofile(f)

    return n_samples

sliced = round(len(res)/4)

# Example: suppose you already have a bitarray
spread_bits = res[:sliced]

# Convert bitarray -> numpy 0/1 array
chips_01 = np.frombuffer(spread_bits.unpack(), dtype=np.uint8)

n_samples = sample_chip_sequence_to_if_file(
    chips_01,
    out_file="gps_signal_if_float32.dat",
    chip_rate=1.023e6,
    fs=4.0e6,
    f_if=1.25e6,
)

print("chips:", len(chips_01))
print("samples:", n_samples)

chips: 191812500
samples: 750000000


#### Multiple satellites

In [7]:
import numpy as np

C_MPS = 299_792_458.0  # speed of light [m/s]


def sample_chip_sequences_to_if_file(
    chip_streams_01,
    distances_m,
    out_file="gps_signal_if_float32.dat",
    chip_rate=1.023e6,
    fs=4.0e6,
    f_if=1.25e6,
    amplitudes=None,
    use_relative_delays=True,
    chunk_size=10_000_000,
):
    """
    chip_streams_01: list of chip arrays, each containing 0/1 values
    distances_m: list/array of satellite-to-receiver distances in meters

    Writes real float32 IF samples.

    IMPORTANT:
    - This version applies delay by zero-padding in sample time.
    - No modulo wrap is used for propagation delay.
    - The whole signal is delayed consistently in receive time.
    """

    if len(chip_streams_01) == 0:
        raise ValueError("Need at least one chip stream")

    if len(distances_m) != len(chip_streams_01):
        raise ValueError("distances_m must have same length as chip_streams_01")

    # Convert chip streams 0/1 -> -1/+1
    chips = []
    for k, chips_01 in enumerate(chip_streams_01):
        arr = np.asarray(chips_01, dtype=np.uint8)
        if not np.all((arr == 0) | (arr == 1)):
            raise ValueError(f"Chip stream {k} contains values other than 0/1")
        chips.append(np.where(arr == 0, -1.0, 1.0).astype(np.float32))

    lengths = [len(c) for c in chips]
    if len(set(lengths)) != 1:
        raise ValueError(f"All chip streams must have same length, got {lengths}")

    n_chips = lengths[0]
    duration = n_chips / chip_rate
    n_signal_samples = int(np.floor(duration * fs))

    if amplitudes is None:
        amplitudes = np.ones(len(chips), dtype=np.float32)
    else:
        amplitudes = np.asarray(amplitudes, dtype=np.float32)
        if len(amplitudes) != len(chips):
            raise ValueError("amplitudes must have same length as chip_streams_01")

    distances_m = np.asarray(distances_m, dtype=np.float64)
    delays_s = distances_m / C_MPS

    if use_relative_delays:
        delays_s = delays_s - np.min(delays_s)

    # Integer-sample delays for now
    delay_samples = np.round(delays_s * fs).astype(np.int64)
    max_delay = int(np.max(delay_samples))

    # Total output must be long enough to include the latest delayed signal
    n_samples_total = n_signal_samples + max_delay

    print("Distances [m]:", distances_m)
    print("Delays [s]:", delays_s)
    print("Delay samples:", delay_samples)
    print("Total output samples:", n_samples_total)

    with open(out_file, "wb") as f:
        for start in range(0, n_samples_total, chunk_size):
            stop = min(start + chunk_size, n_samples_total)

            sample_idx = np.arange(start, stop, dtype=np.int64)
            t = sample_idx.astype(np.float64) / fs

            carrier = np.cos(2 * np.pi * f_if * t).astype(np.float32)
            signal = np.zeros(stop - start, dtype=np.float32)

            for i in range(len(chips)):
                # Map output sample time back to source sample time
                src_sample_idx = sample_idx - delay_samples[i]

                # Valid only where source time is inside the original waveform
                valid = (src_sample_idx >= 0) & (src_sample_idx < n_signal_samples)
                if not np.any(valid):
                    continue

                # Convert valid source sample indices to chip indices
                src_t = src_sample_idx[valid].astype(np.float64) / fs
                chip_phase = src_t * chip_rate
                chip_index = np.floor(chip_phase).astype(np.int64)

                # Clamp to valid chip range
                chip_index = np.clip(chip_index, 0, n_chips - 1)

                baseband = np.zeros(stop - start, dtype=np.float32)
                baseband[valid] = chips[i][chip_index]

                signal += amplitudes[i] * baseband * carrier

            signal.tofile(f)

    return n_samples_total

In [ ]:
# NAV message XOR'ed with C/A
res1=modulo2_frames_runs(1,Z_count_start,1)[:95906250]
res2=modulo2_frames_runs(2,Z_count_start,2)[:95906250]
res3=modulo2_frames_runs(3,Z_count_start,3)[:95906250]
res4=modulo2_frames_runs(4,Z_count_start,4)[:95906250]

res6=modulo2_frames_runs(6,Z_count_start,6)[:95906250]
res7=modulo2_frames_runs(7,Z_count_start,7)[:95906250]
res9=modulo2_frames_runs(9,Z_count_start,9)[:95906250]
res11=modulo2_frames_runs(11,Z_count_start,11)[:95906250]

# Distance to sat
rho1=np.sqrt(sum((ehpm_to_ECEFlocation(1) - ECEF(buddinge[0],buddinge[1],buddinge[2]))**2))
rho2=np.sqrt(sum((ehpm_to_ECEFlocation(2) - ECEF(buddinge[0],buddinge[1],buddinge[2]))**2))
rho3=np.sqrt(sum((ehpm_to_ECEFlocation(3) - ECEF(buddinge[0],buddinge[1],buddinge[2]))**2))
rho4=np.sqrt(sum((ehpm_to_ECEFlocation(4) - ECEF(buddinge[0],buddinge[1],buddinge[2]))**2))
rho6=np.sqrt(sum((ehpm_to_ECEFlocation(6) - ECEF(buddinge[0],buddinge[1],buddinge[2]))**2))
rho7=np.sqrt(sum((ehpm_to_ECEFlocation(7) - ECEF(buddinge[0],buddinge[1],buddinge[2]))**2))
rho9=np.sqrt(sum((ehpm_to_ECEFlocation(9) - ECEF(buddinge[0],buddinge[1],buddinge[2]))**2))
rho11=np.sqrt(sum((ehpm_to_ECEFlocation(11) - ECEF(buddinge[0],buddinge[1],buddinge[2]))**2))

# Unpack to integers (instead of being bits like res)
chips_01 = np.frombuffer(res1.unpack(), dtype=np.uint8)
chips_02 = np.frombuffer(res2.unpack(), dtype=np.uint8)
chips_03 = np.frombuffer(res3.unpack(), dtype=np.uint8)
chips_04 = np.frombuffer(res4.unpack(), dtype=np.uint8)

chips_06 = np.frombuffer(res6.unpack(), dtype=np.uint8)
chips_07 = np.frombuffer(res7.unpack(), dtype=np.uint8)
chips_09 = np.frombuffer(res9.unpack(), dtype=np.uint8)
chips_11 = np.frombuffer(res11.unpack(), dtype=np.uint8)

distances_m = [
    rho1,  # satellite 1 range to receiver
    rho2,  # satellite 2 range to receiver
    rho3,  # satellite 3 range to receiver
    rho4,  # satellite 4 range to receiver
    rho6,  # satellite 6 range to receiver
    rho7,  # satellite 7 range to receiver
    rho9,  # satellite 9 range to receiver
    rho11,  # satellite 11 range to receiver
]

n_samples = sample_chip_sequences_to_if_file(
    [chips_01, chips_02, chips_03, chips_04,chips_06,chips_07,chips_09,chips_11],
    distances_m = distances_m,
    out_file = "gps_signal4_if_float32.dat",
    chip_rate = 1.023e6,
    fs = 4.0e6,
    f_if = 1.25e6,
    amplitudes = [0.7, 0.8, 0.6, 0.75,0.7, 0.8, 0.6, 0.75],
    use_relative_delays = False,
)

print("samples:", n_samples)

Distances [m]: [21538587.64452836 24205925.19551226 20054507.79143932 21513328.22316682
 21290782.26494222 25389770.94829089 23269387.89892859 25575579.44677637]
Delays [s]: [0.07184499 0.08074228 0.06689464 0.07176074 0.07101841 0.08469116
 0.07761832 0.08531095]
Delay samples: [287380 322969 267579 287043 284074 338765 310473 341244]
Total output samples: 375341244
samples: 375341244
